In [27]:
import pandas as pd
import joblib
import numpy as np


interactions = pd.read_parquet("data\\interactions.parquet")

metadata = joblib.load("data\\metadata.pkl")

interactions

,user_idx,movie_idx,rating
0,0,1104,5.0
1,0,639,3.0
2,0,853,3.0
3,0,3177,4.0
4,0,2162,5.0
...,...,...,...
1000204,6039,1019,1.0
1000205,6039,1022,5.0
1000206,6039,548,5.0
1000207,6039,1024,4.0


### Test Matrix Factorization

In [28]:
n_movies = metadata["n_movies"]
n_users = metadata["n_users"]

In [29]:
n_factors = 20
lr = 0.01
reg = 0.02
epochs = 20

In [30]:
train_parts = []
test_parts = []

for _, group in interactions.groupby("user_idx"):
    train_part = group.sample(frac=0.8, random_state=42)
    test_part = group.drop(train_part.index)

    train_parts.append(train_part)
    test_parts.append(test_part)

train = pd.concat(train_parts).reset_index(drop=True)
test = pd.concat(test_parts).reset_index(drop=True)

In [31]:
print(train)
print(test)

        user_idx  movie_idx  rating
0              0       2592     4.0
1              0       1781     5.0
2              0       1117     4.0
3              0       2205     4.0
4              0       2488     4.0
...          ...        ...     ...
800188      6039       2246     4.0
800189      6039        971     3.0
800190      6039       1563     3.0
800191      6039       1199     3.0
800192      6039        148     2.0

[800193 rows x 3 columns]
        user_idx  movie_idx  rating
0              0       2599     5.0
1              0        581     5.0
2              0        970     5.0
3              0       2889     5.0
4              0       2128     3.0
...          ...        ...     ...
200011      6039       1841     3.0
200012      6039       1848     5.0
200013      6039       1018     3.0
200014      6039       1024     4.0
200015      6039       1025     4.0

[200016 rows x 3 columns]


In [ ]:
def train_(train, n_users, n_movies, n_factor, lr, reg, epochs=20, show=False, random_state=42):

    rng = np.random.default_rng(random_state)

    P = rng.normal(
        0,
        0.1,
        size=(n_users, n_factor),
    )

    Q = rng.normal(
        0,
        0.1,
        size=(n_movies, n_factor),
    )

    mu = train.rating.mean()

    bu = np.zeros(n_users)
    bi = np.zeros(n_movies)

    for epoch in range(epochs):
        errors = []

        for row in train.itertuples(index=False):

            u = row.user_idx
            m = row.movie_idx
            r = row.rating

            prediction = (
                mu + bu[u] + bi[m] + P[u] @ Q[m]
            )

            error = r - prediction

            errors.append(error)

            old_p = P[u].copy()

            bu[u] += lr * (
                error - reg * bu[u]
            )

            bi[m] += lr * (
                error - reg * bi[m]
            )

            P[u] += lr * (
                error * Q[m] - reg * P[u]
            )

            Q[m] += lr * (
                error * old_p - reg * Q[m]
            )

        if show:
            rmse = np.sqrt(
                np.mean(
                    np.square(errors)
                )
            )
            print(
                f"Epoch {epoch + 1}: "
                f"RMSE {rmse:.4}: "
            )

    return P, Q, mu, bu, bi

In [33]:
def recommend(user_idx, P, Q, mu, bu, bi, k=10):

    watched = (
        train.loc[
            train.user_idx == user_idx,
            "movie_idx"
        ]
    ).unique()

    scores = (
        mu + bu[user_idx] + bi + P[user_idx] @ Q.T
        )

    scores[watched] = -np.inf

    recomemend = np.argsort(scores)[::-1][:k]

    return [
        (movie_idx, scores[movie_idx])
        for movie_idx in recomemend
    ]


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.metrics.metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
)

def metrics(P, Q, mu, bu, bi, val, k=10):

    recalls = []
    precisions = []
    ndcgs = []

    for user_idx in val.user_idx.unique():

        recommendations = recommend(user_idx, P, Q, mu, bu, bi, k)
        val_user = val[val.user_idx == user_idx]

        recommended = [
            movie_idx
            for movie_idx, _ in recommendations
        ]

        relevant = val_user.loc[
            val_user.rating >= 4,
            "movie_idx"
        ].tolist()

        if len(relevant) == 0:
            continue

        recalls.append(recall_at_k(relevant, recommended, k))
        precisions.append(precision_at_k(relevant, recommended, k))
        ndcgs.append(ndcg_at_k(relevant, recommended, k))

    return (
        np.mean(recalls),
        np.mean(precisions),
        np.mean(ndcgs)
    )

### Grid Search


In [35]:
from itertools import product

param_grid = {
    "n_factors": [5, 10, 20, 40],
    "lr": [0.005, 0.01, 0.02],
    "reg": [0.01, 0.02, 0.05],
}

In [ ]:
def grid_search(param_grid, train, val, n_users, n_movies, show):

    res = []

    i = 0

    for n_factor, lr, reg in product(
        param_grid["n_factors"],
        param_grid["lr"],
        param_grid["reg"]
    ):
        i += 1

        print(f"attempt: {i}")
        print(f"n_factors={n_factor}, lr={lr}, reg={reg}")

        P, Q, mu, bu, bi = train_(train, n_users, n_movies, n_factor, lr, reg, show=show)

        recall, precision, ndcg = metrics(P, Q, mu, bu, bi, val)
        
        res.append({
            "n_factors": n_factor,
            "lr": lr,
            "reg": reg,
            "recall": recall,
            "precision": precision,
            "ndcg": ndcg
        })

    return res


In [ ]:
grid_search(param_grid, train, n_users, n_movies, True)

In [24]:
res_df = pd.DataFrame(res)
res_df

,n_factors,lr,reg,recall,precision,ndcg
0,5,0.005,0.01,0.041755,0.075004,0.082822
1,5,0.005,0.02,0.039195,0.070633,0.078802
2,5,0.005,0.05,0.030322,0.053631,0.056500
3,5,0.010,0.01,0.047565,0.076583,0.091866
4,5,0.010,0.02,0.044736,0.075187,0.087846
5,5,0.010,0.05,0.035957,0.060246,0.067603
6,5,0.020,0.01,0.038689,0.061260,0.071217
7,5,0.020,0.02,0.035932,0.058601,0.066070
8,5,0.020,0.05,0.031218,0.048745,0.052235
9,10,0.005,0.01,0.043298,0.076533,0.086759


Сделав подбор гиперпараметров и отсортировав по ndcg можно сделать вывод что лучшие значения это 
<center>n_factors = 40 </center>
<center>lr        = 0.005 </center>
<center>reg       = 0.01 </center>

In [ ]:
res_df.to_parquet(
    "data\\matrix_factorization\\01_parametres.parquet",
    index= False
)


### Best Parametres

In [8]:
n_factors = 40
lr        = 0.005
reg       = 0.01
epochs    = 20

Можно улучшить модель, сделав подбор вокруг получившизся параметров, но я не буду этого делать

In [ ]:
epochs = [5, 10, 20, 40, 80]

res = []

for epoch in epochs:

    print(f"epoch: {epoch}")

    P, Q, mu, bu, bi = train_(train, n_users, n_movies, n_factors, lr, reg, epoch, show)
    
    recall, precision, ndcg = metrics(P, Q, mu, bu, bi)

    res.append({
        "epoch": epoch,
        "recall": recall,
        "precision": precision,
        "ndcg": ndcg
    })

epoch: 5
Epoch 1: RMSE 0.9839: 
Epoch 2: RMSE 0.9245: 
Epoch 3: RMSE 0.9097: 
Epoch 4: RMSE 0.9014: 
Epoch 5: RMSE 0.8951: 
recall at k: 0.028622484097581658
precision at k: 0.04831311284693369
ndcg at k: 0.05392131864954028
epoch: 10
Epoch 1: RMSE 0.9838: 
Epoch 2: RMSE 0.9244: 
Epoch 3: RMSE 0.9096: 
Epoch 4: RMSE 0.9013: 
Epoch 5: RMSE 0.895: 
Epoch 6: RMSE 0.889: 
Epoch 7: RMSE 0.8822: 
Epoch 8: RMSE 0.8741: 
Epoch 9: RMSE 0.8643: 
Epoch 10: RMSE 0.8534: 
recall at k: 0.02981714630098598
precision at k: 0.05625727106531494
ndcg at k: 0.06363721286454221
epoch: 20
Epoch 1: RMSE 0.9837: 
Epoch 2: RMSE 0.9242: 
Epoch 3: RMSE 0.9094: 
Epoch 4: RMSE 0.901: 
Epoch 5: RMSE 0.8944: 
Epoch 6: RMSE 0.8879: 
Epoch 7: RMSE 0.8806: 
Epoch 8: RMSE 0.8719: 
Epoch 9: RMSE 0.862: 
Epoch 10: RMSE 0.8512: 
Epoch 11: RMSE 0.8401: 
Epoch 12: RMSE 0.8289: 
Epoch 13: RMSE 0.8176: 
Epoch 14: RMSE 0.8065: 
Epoch 15: RMSE 0.7956: 
Epoch 16: RMSE 0.785: 
Epoch 17: RMSE 0.7748: 
Epoch 18: RMSE 0.7651: 
Epoch 

In [18]:
print(pd.DataFrame(res))

   epoch    recall  precision      ndcg
0      5  0.028622   0.048313  0.053921
1     10  0.029817   0.056257  0.063637
2     20  0.047909   0.086389  0.101748
3     40  0.042684   0.071680  0.084780
4     80  0.028755   0.049277  0.055765


По получившимся данным прекрасно видно что 20 эпох лучши вариант

### Final Parametres


In [19]:
n_factors = 40
lr        = 0.005
reg       = 0.01
epochs    = 20

In [20]:
P, Q, mu, bu, bi = train_(train, n_users, n_movies, n_factors, lr, reg, epochs, True)

recall, precision, ndcg = metrics(P, Q, mu, bu, bi)

print(
    f"Recall@10: {recall:.4f}\n"
    f"Precision@10: {precision:.4f}\n"
    f"NDCG@10: {ndcg:.4f}"
)

Epoch 1: RMSE 0.9839: 
Epoch 2: RMSE 0.9244: 
Epoch 3: RMSE 0.9096: 
Epoch 4: RMSE 0.9013: 
Epoch 5: RMSE 0.895: 
Epoch 6: RMSE 0.889: 
Epoch 7: RMSE 0.8822: 
Epoch 8: RMSE 0.8741: 
Epoch 9: RMSE 0.8645: 
Epoch 10: RMSE 0.8538: 
Epoch 11: RMSE 0.8427: 
Epoch 12: RMSE 0.8314: 
Epoch 13: RMSE 0.82: 
Epoch 14: RMSE 0.8087: 
Epoch 15: RMSE 0.7976: 
Epoch 16: RMSE 0.7868: 
Epoch 17: RMSE 0.7763: 
Epoch 18: RMSE 0.7664: 
Epoch 19: RMSE 0.7569: 
Epoch 20: RMSE 0.748: 
recall at k: 0.04580826932225751
precision at k: 0.08155226857237825
ndcg at k: 0.09539537801567535
Recall@10: 0.0458
Precision@10: 0.0816
NDCG@10: 0.0954


### With val parts

In [ ]:
train_parts = []
test_parts = []
val_parts = []

for _, group in interactions.groupby("user_idx"):
    train_part = group.sample(frac=0.8, random_state=42)

    remaining = group.drop(train_part.index)

    val_part = remaining.sample(frac=0.5, random_state=42)

    test_part = remaining.drop(val_part.index)

    train_parts.append(train_part)
    val_parts.append(val_part)
    test_parts.append(test_part)

train = pd.concat(train_parts).reset_index(drop=True)
test = pd.concat(test_parts).reset_index(drop=True)
val = pd.concat(val_parts).reset_index(drop=True)

,user_idx,movie_idx,rating
0,0,1178,5.0
1,0,2599,5.0
2,0,593,4.0
3,0,1154,4.0
4,0,970,5.0
...,...,...,...
99861,6039,859,3.0
99862,6039,1106,3.0
99863,6039,2775,1.0
99864,6039,3190,3.0


In [ ]:
res = grid_search(param_grid, train, val, n_users, n_movies, True)

attempt: 1
n_factors=5, lr=0.005, reg=0.01
Epoch 1: RMSE 0.9822: 
Epoch 2: RMSE 0.9259: 
Epoch 3: RMSE 0.9138: 
Epoch 4: RMSE 0.9083: 
Epoch 5: RMSE 0.9051: 
Epoch 6: RMSE 0.903: 
Epoch 7: RMSE 0.9012: 
Epoch 8: RMSE 0.8996: 
Epoch 9: RMSE 0.8977: 
Epoch 10: RMSE 0.8953: 
Epoch 11: RMSE 0.892: 
Epoch 12: RMSE 0.8877: 
Epoch 13: RMSE 0.8825: 
Epoch 14: RMSE 0.8768: 
Epoch 15: RMSE 0.8713: 
Epoch 16: RMSE 0.8662: 
Epoch 17: RMSE 0.8615: 
Epoch 18: RMSE 0.8572: 
Epoch 19: RMSE 0.8533: 
Epoch 20: RMSE 0.8498: 
[0.0, 0.0, 0.0, 0.0, 0.1111111111111111, 0.0, 0.0, 0.0, 0.16666666666666666, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.047619047619047616, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.12903225806451613, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.125, 0.0, 0.0, 0.23529411764705882, 0.0, 0.0, 0.0, 0.0, 0.018867924528301886, 0.0, 0.0, 0.0, 0.0, 0.03225806451612903, 0.0, 0.0, 0.0, 0.0, 0.25, 0.0, 0.09090909090909091, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

KeyboardInterrupt: 

: 

In [ ]:
res_df = pd.DataFrame(res)

,n_factors,lr,reg,recall,precision,ndcg
0,5,0.005,0.01,0.0,0.0,0.0
1,5,0.005,0.02,0.0,0.0,0.0
2,5,0.005,0.05,0.0,0.0,0.0
3,5,0.010,0.01,0.0,0.0,0.0
4,5,0.010,0.02,0.0,0.0,0.0
5,5,0.010,0.05,0.0,0.0,0.0
6,5,0.020,0.01,0.0,0.0,0.0
7,5,0.020,0.02,0.0,0.0,0.0
8,5,0.020,0.05,0.0,0.0,0.0
9,10,0.005,0.01,0.0,0.0,0.0


In [ ]:
res_df.to_parquet(
    "data\\matrix_factorization\\02_parametres.parquet",
    index= False
)
